# Hybrid physics+data baseline — advisor follow-up review, point 4

Combines the physics-informed objective (Pi = U - W, identical to the real PI training) with the data-driven relative-L2 loss, both from ONE forward pass (so they see the same dropout mask). Pi and the data loss live on very different scales, so Pi is rescaled by a running EMA of its own magnitude before combining (`--hybrid_phys_norm running`, the default) at weight `--hybrid_phys_weight` (default 0.5).

* Run on the **representative case** (B1 x Neo-Hookean), per the advisor's own wording.
* Needs the SAME dataset the other runs used (the labels are used for the data term; no new FEM solves).
* `history.json` logs `phys_raw`, `phys_term` (normalized), and `data_term` separately at every eval step — inspect these if the combined loss looks off-balance.
* Compare `best_val_rel_L2` against the pure physics-informed run, the pure data-driven run, and the Sobolev run.

In [ ]:
# =====================================================================
#  CELL — Hybrid physics+data operator (advisor follow-up, point 4)
#  GPU strongly preferred.
# =====================================================================
import os, subprocess, sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
                    'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - this will be slow, prefer Runtime > Change runtime type > GPU')

R = '/content/drive/MyDrive/pfem_run'
OUT = f'{R}/hybrid_pi_dd'
os.makedirs(OUT, exist_ok=True)

# --- the representative case, per the advisor's own wording ---
GEOMETRY  = 'B1'
MATERIAL  = 'neo_hookean'
DATA      = f'{R}/results/datasets/B1_neo_hookean/hyperelastic_training_data_q4.npz'
OPT_STEPS = 75000
BATCH     = 8
PHYS_WEIGHT = 0.5
PHYS_NORM   = 'running'   # or 'none' for the raw, unnormalized combination
# --------------------------------------------------------------------------

assert os.path.exists(DATA), (
    f'dataset not found: {DATA}\n'
    'Check the path -- it must be the SAME .npz every other run in this '
    'comparison used.')
print('dataset:', DATA, f'({os.path.getsize(DATA)/1e6:.1f} MB)')

run([
    sys.executable, '-u', '-m', 'omar_pfem.train_data_driven',
    '--geometry', GEOMETRY, '--material', MATERIAL,
    '--path', DATA,
    '--ntrain', '800', '--ntest', '200',
    '--batch_size', str(BATCH),
    '--opt_steps', str(OPT_STEPS),
    '--loss', 'hybrid',
    '--hybrid_phys_weight', str(PHYS_WEIGHT),
    '--hybrid_phys_norm', PHYS_NORM,
    '--eval_every', '2000',
    '--out_dir', f'{OUT}/{GEOMETRY}_{MATERIAL}',
])

print(f"\nDone. Result: {OUT}/{GEOMETRY}_{MATERIAL}/data_driven_{GEOMETRY}_{MATERIAL}.json")
print('Compare best_val_rel_L2 against pure PI (Table 5), pure DD (--loss rel_l2),')
print('and the Sobolev run. history.json logs phys_raw/phys_term/data_term separately')
print('at every eval step -- check these before trusting the combined number if the')
print('two terms look wildly unbalanced.')